# Píldora 4 - Introducción a MLflow Tracking

Objetivo: registrar experimentos de Machine Learning con parámetros, métricas y artefactos.

> Esta práctica guiada es opcional y sirve de puente entre la demo y el reto Orden 66.

---

**Ruta de la píldora:** demo guiada → reto `03_Orden_66_RETO.ipynb` → corrección con `02_Archivos_Jedi_SOLUCION.ipynb`.
**Entorno validado:** Python 3.12, MLflow 3.14.0, scikit-learn 1.9.0, pandas 2.3.3 y matplotlib 3.11.0.


## 0. Preparar el entorno

In [ ]:
# Entorno validado el 15-07-2026. Ejecuta esta celda una sola vez en Colab.
%pip install -q "mlflow==3.14.0" "scikit-learn==1.9.0" "pandas==2.3.3" "matplotlib==3.11.0"

## 1. Importaciones y datos

Usaremos el dataset **Digits**, formado por imágenes de números escritos a mano. El modelo tendrá que reconocer qué dígito aparece, del 0 al 9.

In [ ]:
from pathlib import Path

import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from mlflow.models import infer_signature
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

In [ ]:
data = load_digits()

X_train, X_test, y_train, y_test = train_test_split(
    data.data,
    data.target,
    test_size=0.2,
    random_state=42,
    stratify=data.target,
)

tracking_db = Path("mlflow_jedi_digits.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")
mlflow.set_experiment("Biblioteca_Templo_Jedi_Digits_Practica")

print("Muestras:", data.data.shape)
print("Clases:", list(data.target_names))

plt.imshow(data.images[0], cmap="gray")
plt.title(f"Dígito real: {data.target[0]}")
plt.axis("off")
plt.show()

## 2. Ejercicio 1 - Primera run

Cambia `n_estimators` y `max_depth`, ejecuta la celda y comprueba la run en MLflow UI.


In [ ]:
n_estimators = 100
max_depth = 10

with mlflow.start_run(run_name="primer_tracking"):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("dataset", "digits")
    mlflow.log_metric("accuracy", accuracy)

    print(f"Accuracy: {accuracy:.4f}")

## 3. Ejercicio 2 - Comparar varias runs

Ejecuta varias configuraciones y compara los resultados en MLflow UI.


In [ ]:
configs = [
    {"n_estimators": 30, "max_depth": 5},
    {"n_estimators": 50, "max_depth": 8},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": 12},
    {"n_estimators": 300, "max_depth": None},
]

for config in configs:
    run_name = f"rf_{config['n_estimators']}_{config['max_depth']}"

    with mlflow.start_run(run_name=run_name):
        model = RandomForestClassifier(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            random_state=42,
        )
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        mlflow.log_params({
            **config,
            "dataset": "digits",
            "test_size": 0.2,
            "random_state": 42,
        })
        mlflow.log_metric("accuracy", accuracy)

        print(run_name, "accuracy:", round(accuracy, 4))

## 4. Ejercicio 3 - Registrar un artefacto

Guarda una matriz de confusion como imagen y registrala como artefacto.


In [ ]:
config = {"n_estimators": 100, "max_depth": 10}

with mlflow.start_run(run_name="run_con_artefacto"):
    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        random_state=42,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    mlflow.log_params({
        **config,
        "dataset": "digits",
        "test_size": 0.2,
        "random_state": 42,
    })
    mlflow.log_metric("accuracy", accuracy)

    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test,
        display_labels=data.target_names,
        cmap="Blues",
    )
    plt.title("Matriz de confusión · Digits")
    artifact_path = "matriz_confusion_digits.png"
    plt.savefig(artifact_path, dpi=150, bbox_inches="tight")
    plt.close()

    mlflow.log_artifact(artifact_path)
    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(
        sk_model=model,
        name="modelo_jedi",
        signature=signature,
        input_example=X_train[:3],
    )

    print(f"Run con artefacto registrada. Accuracy: {accuracy:.4f}")

## 5. Preguntas finales

1. ¿Qué diferencia hay entre parámetro y métrica?
2. ¿Por qué una matriz de confusión puede ser más informativa que la accuracy?
3. ¿Qué dígitos confunde con más frecuencia el modelo?
4. ¿Qué guardarías como artefacto en un proyecto de clasificación real?
5. ¿Qué configuración elegirías y cómo la justificarías?